# FloodLite D4: ground-truth-gated distillation on the combined corpus

Trains **3 students x {gated} x 1 fold** on *flood-combined-fsegnet*, reusing the
D2 teacher checkpoint of that fold (set `FOLD` below; run once per fold 0-2).

Gated KD = combined response+feature KD where both KD terms are masked by the
per-pixel teacher-correctness map (`floodlite.losses.teacher_correct_mask`), so a
teacher false alarm on a dry scene carries no distillation signal. Everything else
(alpha=0.5, beta=0.1, T=4, lr 5e-4, 35 epochs, batch 8, seed 42, source-balanced
sampler) is identical to the D2 `comb` runs.

**Prerequisites**
1. *Add Data*: `flood-combined-fsegnet` (data) and `floodlite-combined-ckpts`
   (D2 checkpoints incl. `teacher_fold{0,1,2}.pt`), both by tasfianoorchowdhury.
2. GPU on (T4). One fold = 3 student runs (about 4-5 h).
3. The code is cloned from the `resubmission-jrtip` branch (gated KD lives there).


In [ ]:
FOLD = 0          # <-- set to 0, 1, or 2 (one fold per session)
EPOCHS = 35
BATCH = 8


In [ ]:
!git clone --depth 1 -b resubmission-jrtip https://github.com/NoorTasfia/flood-segmentation.git /kaggle/working/repo
%cd /kaggle/working/repo/code
%pip install -q "segmentation_models_pytorch>=0.4.0,<0.6" "timm>=0.9.7,<2.0" albumentations==1.4.18 torchprofile
!python -m pytest tests/test_losses_gated.py -q 2>&1 | tail -2


In [ ]:
from pathlib import Path
import shutil
# Data root = the folder that holds manifest.csv
candidates = list(Path('/kaggle/input').glob('**/manifest.csv'))
assert candidates, 'Attach the flood-combined-fsegnet dataset first (Add Data).'
DATA_ROOT = candidates[0].parent
print('DATA_ROOT =', DATA_ROOT)

# Teacher checkpoint of this fold from the D2 checkpoint dataset (never retrained here)
OUT = Path('/kaggle/working/runs_gated'); OUT.mkdir(parents=True, exist_ok=True)
t_ckpts = list(Path('/kaggle/input').glob(f'**/teacher_fold{FOLD}.pt'))
assert t_ckpts, f'Attach floodlite-combined-ckpts: teacher_fold{FOLD}.pt not found.'
shutil.copy(t_ckpts[0], OUT / f'teacher_fold{FOLD}.pt')
print('teacher checkpoint:', t_ckpts[0])

# Resume support: gated student checkpoints from a previous session of this kernel
prev = list(Path('/kaggle/input').glob('**/runs_gated/*_gated_fold*.pt'))
for p in prev:
    shutil.copy(p, OUT / p.name)
if prev:
    print(f'Resumed {len(prev)} gated checkpoints')


In [ ]:
!python scripts/run_full_experiment.py \
    --dataset combined \
    --data_root {DATA_ROOT} \
    --fold {FOLD} --folds 1 \
    --kd_configs gated \
    --skip_quant --skip_if_exists \
    --epochs_teacher {EPOCHS} --epochs_student {EPOCHS} \
    --batch_size {BATCH} \
    --out_dir /kaggle/working/runs_gated


In [ ]:
# Keep results + checkpoints as session output
import json
from pathlib import Path
res = Path('/kaggle/working/runs_gated/results.json')
if res.exists():
    r = json.loads(res.read_text())
    for fold, blocks in r.items():
        print(f"== {fold} ==")
        t = blocks['teacher']
        print(f"  teacher test: IoU {t['test']['iou_mean']:.4f}  FAR {t['test'].get('false_alarm_rate', float('nan')):.3f}")
        for name, m in blocks['students'].items():
            print(f"  {name}: val {m['val']['iou_mean']:.4f}  test {m['test']['iou_mean']:.4f} "
                  f"flood {m['test'].get('iou_flood_mean', float('nan')):.4f} "
                  f"FAR {m['test'].get('false_alarm_rate', float('nan')):.3f}  ext {m['external']['iou_mean']:.4f}")
